# 拍照（在 JetBot 上跑）

**下一格執行後，左邊是即時畫面、右邊是拍照按鈕。** 先選「任務」，再填兩個欄位，
檔名、資料夾、格式都會自動照規則設好，按鈕上方會先顯示下一張的檔名。

| 任務 | 資料夾 | 檔名 | 格式 |
|---|---|---|---|
| 色塊校正 | `samples/` | `close_01.png`、`1.5m_01.png`、`bg_01.png` | PNG |
| 工作一 警示物件 | `raw/task_A_alert/` | `alert_地點_時段_0001.jpg` | JPG |
| 工作三 跌倒姿態 | `raw/task_C_fall/` | `fall_志工_姿態_0001.jpg` | JPG |

**工作二（跟隨標記）不在這裡**：已定案由 HSV 辨識腳踝布條、不讓 YOLO 學 `marker`，
布條照片就是「色塊校正」那組，不需要另外拍 YOLO 用的 500～800 張。

**工作一、三的流水號是 4 位數，而且整個資料夾共用一組**（依《影像蒐集工作分配表》第五節）：
換志工、換情境都不會歸零，接著資料夾裡最大的號碼往下編。多人輪流拍也不會撞檔名。

色塊校正維持原本的規則（每個前綴各自 2 位數編號），`analyze_marker.py` 靠前綴裡的 `1.5m` 判斷距離。

### 拍之前先確認

1. **攝影機角度已經固定**
2. **解析度 640×480（4:3）**，不裁切不縮放
3. **SD 卡空間**：一張 PNG 約 450 KB、JPG 約 50 KB。下一格會顯示剩餘空間

拍完跑最後一格把相機釋放掉。

In [ ]:
import os
import re
import shutil

import cv2
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

# 640x480(4:3):感測器原生比例(816x616),不會被壓扁;與上車推論的相機設定一致。
CAM_W, CAM_H = 640, 480
camera = Camera.instance(width=CAM_W, height=CAM_H)

# --- 任務設定(依《影像蒐集工作分配表》第五節)---------------------------------
# 欄位 A、B 的建議值只是提示,可以自己填別的,但不能有底線或空白
# (底線是欄位的分隔符號,多一個底線檔名就拆不回來了)。
TASKS = {
    '色塊校正': dict(folder='samples', prefix=None, ext='.png', digits=2,
                     a=('前綴', 'close', 'close / 0.5m / 0.8m / 1.5m / 2.0m / bg'),
                     b=None),
    '工作一 警示物件': dict(folder='raw/task_A_alert', prefix='alert', ext='.jpg', digits=4,
                            a=('地點', 'road', 'gate(校門口) / road(臨溪路) / bg(純背景)'),
                            b=('時段', 'am', 'am / noon / pm')),
    # 工作二(跟隨標記)已定案改由 HSV 做,不需要 YOLO 用的照片,所以沒有這個選項。
    '工作三 跌倒姿態': dict(folder='raw/task_C_fall', prefix='fall', ext='.jpg', digits=4,
                            a=('志工', 'v1', 'v1 / v2 / v3 …'),
                            b=('姿態', 'sideways', 'sideways / facedown / faceup / kneeling;沒綁布條加 -nomark,例如 sideways-nomark')),
}

# --- 左邊:即時畫面 --------------------------------------------------------
# 顯示 480x360(原圖的 3/4),旁邊還放得下按鈕;存檔一律是 640x480 原圖。
image_widget = widgets.Image(format='jpeg', width=480, height=360)
camera_link = traitlets.dlink((camera, 'value'), (image_widget, 'value'),
                              transform=bgr8_to_jpeg)

# --- 右邊:控制項 ----------------------------------------------------------
task_widget = widgets.Dropdown(options=list(TASKS), value='色塊校正', description='任務')
a_widget = widgets.Text(description='欄位 A')
b_widget = widgets.Text(description='欄位 B')
hint = widgets.HTML()
next_label = widgets.HTML()
snap_button = widgets.Button(description='拍一張', button_style='success',
                             layout=widgets.Layout(width='140px', height='60px'))
undo_button = widgets.Button(description='刪掉上一張', button_style='danger')
status = widgets.HTML()      # 拍照結果直接顯示在按鈕下方,不用捲動找 print

_last = {'path': None}


def _task():
    return TASKS[task_widget.value]


def _folder():
    folder = _task()['folder']
    if not os.path.isdir(folder):
        os.makedirs(folder)
    return folder


def _field(widget, name):
    v = (widget.value or '').strip()
    if not v:
        raise ValueError('%s 不能空白' % name)
    if re.search(r'[\s_]', v):
        raise ValueError('%s 不能有底線或空白:「%s」' % (name, v))
    return v


def _stem():
    """回傳不含流水號的檔名開頭,例如 marker_v1_walk。欄位不合法會丟 ValueError。"""
    t = _task()
    a = _field(a_widget, t['a'][0])
    if t['prefix'] is None:                  # 色塊校正:前綴就是欄位 A
        return a
    b = _field(b_widget, t['b'][0])
    return '%s_%s_%s' % (t['prefix'], a, b)


def _next_path():
    t = _task()
    folder = _folder()
    stem = _stem()
    names = os.listdir(folder)
    if t['digits'] == 4:
        # 工作一、三:整個資料夾共用流水號,取資料夾裡最大的 4 位數 +1
        nums = [int(m.group(1)) for m in
                (re.search(r'_(\d{4})\.(?:jpg|jpeg|png)$', n, re.I) for n in names) if m]
        n = max(nums) + 1 if nums else 1
        return os.path.join(folder, '%s_%04d%s' % (stem, n, t['ext']))
    # 色塊校正:每個前綴各自編號(2 位數)
    n = 1
    while any(x.startswith('%s_%02d.' % (stem, n)) for x in names):
        n += 1
    return os.path.join(folder, '%s_%02d%s' % (stem, n, t['ext']))


def _refresh(msg=''):
    folder = _folder()
    n = len([f for f in os.listdir(folder)
             if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    free_gb = shutil.disk_usage(folder).free / 1e9
    warn = ' <b style="color:#c00">空間不足!</b>' if free_gb < 0.3 else ''
    try:
        nxt = '下一張:<b>%s</b>' % os.path.basename(_next_path())
    except ValueError as e:
        nxt = '<b style="color:#c00">%s</b>' % e
    next_label.value = nxt
    status.value = ('%s<br>%s/ 共 <b>%d</b> 張 · SD 卡剩 %.2f GB%s'
                    % (msg, folder, n, free_gb, warn))


def _on_task(_change=None):
    t = _task()
    a_widget.description, a_widget.value = t['a'][0], t['a'][1]
    if t['b'] is None:
        b_widget.layout.display = 'none'
        hint.value = '<small>%s:%s</small>' % (t['a'][0], t['a'][2])
    else:
        b_widget.layout.display = None
        b_widget.description, b_widget.value = t['b'][0], t['b'][1]
        hint.value = '<small>%s:%s<br>%s:%s</small>' % (
            t['a'][0], t['a'][2], t['b'][0], t['b'][2])
    _refresh('切換到「%s」' % task_widget.value)


def save_one(_):
    try:
        path = _next_path()
    except ValueError as e:
        _refresh('<b style="color:#c00">沒拍:%s</b>' % e)
        return
    # camera.value 就是 BGR8 的 numpy 陣列,imwrite 直接吃,不要先轉色彩空間
    params = [cv2.IMWRITE_JPEG_QUALITY, 95] if path.endswith('.jpg') else []
    cv2.imwrite(path, camera.value, params)
    _last['path'] = path
    _refresh('✅ 存了 <b>%s</b>' % os.path.basename(path))


def undo_one(_):
    path = _last['path']
    if path and os.path.exists(path):
        os.remove(path)
        _last['path'] = None
        _refresh('🗑️ 刪掉 %s' % os.path.basename(path))
    else:
        _refresh('沒有可刪的(只能刪剛拍的那一張)')


snap_button.on_click(save_one)
undo_button.on_click(undo_one)
task_widget.observe(_on_task, names='value')
a_widget.observe(lambda _c: _refresh(), names='value')
b_widget.observe(lambda _c: _refresh(), names='value')
_on_task()

controls = widgets.VBox([task_widget, a_widget, b_widget, hint, next_label,
                         widgets.HBox([snap_button, undo_button]), status],
                        layout=widgets.Layout(padding='0 0 0 12px'))
display(widgets.HBox([image_widget, controls]))

### 拍攝清單(色塊校正)

每個情境改一次「檔名前綴」再按上面的拍照按鈕，檔名會自動編號。

**量顏色（前綴 `close`）—— 5~8 張就夠**
- 一般室內燈光下 2 張
- 靠窗、光線比較亮 2 張
- 陰影裡、光線比較暗 2 張
- 貼紙稍微傾斜（模擬走路時腿的角度）2 張

光線變化一定要涵蓋，不然門檻會只在拍攝當下那個亮度成立。

**驗證距離（前綴 `0.5m` / `0.8m` / `1.5m` / `2.0m`）—— 每個距離 3~4 張**
- 目標者站定、正面 1 張
- 側身 1 張
- 走路中途（兩腿交錯的那一瞬間）1~2 張 ← **最重要，這是最容易失敗的狀況**

**背景（前綴 `bg`）—— 3~5 張**
- 實際展示場地、有路人但**沒人貼布條**，背景有樹叢更好

**如果霧面和亮面兩種材質都買了**，各拍一輪，前綴加上 `matte` / `gloss`
（例如 `gloss_close`、`matte_1.5m`），分開跑 `--pick` 就能直接比過曝比例。

### 收尾

拍完一定要跑下面這格把相機釋放掉，否則其他 notebook（包括 `live_demo_following_sm.ipynb`）
會開不起來，錯誤訊息通常看不出是這個原因。

萬一忘了跑就關掉 notebook，到終端機執行 Kernel → Restart，或重開 JetBot 的 Jupyter。

In [ ]:
camera_link.unlink()
camera.stop()
print('相機已釋放。要繼續拍請從第一格重跑。')